In [18]:
from debugpy.common.log import timestamp_format
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Markdown
import pandas as pd
import time
from rag_flow import RAGFlow
from eval import Eval
from datetime import datetime


In [19]:
def load_model():
    model_name = "Qwen/Qwen3-8B"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True #swith it off to download from hugging face if you don't have it locally
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
    return {"model": model, "tokenizer": tokenizer}


In [20]:
model_and_tokenizer = load_model()
model = model_and_tokenizer["model"]
tokenizer = model_and_tokenizer["tokenizer"]

Loading weights: 100%|██████████| 399/399 [00:03<00:00, 117.12it/s]


In [21]:
params = {
    "model": model,
    "tokenizer": tokenizer,
    "web_query_max_new_t": 50,
    "titles_limit": 5,
    "top_k_titles": 5,
    "chunk_size": 200,
    "chunk_overlap_fraction": 0.2,
    "top_k_chunks": 5,
    "answer_max_new_t": 500
}


rag = RAGFlow(params)

answer = rag.run("How does retrieval-augmented generation reduce hallucinations in large language models??")


2026-10-06 16:17:47,801 | INFO | user prompt: How does retrieval-augmented generation reduce hallucinations in large language models??, web query: retrieval augmented generation hallucinations large language models
Batches: 100%|██████████| 12/12 [00:01<00:00, 11.45it/s]
2026-10-06 16:17:53,077 | INFO | Answer generation input tokens: 2253


In [22]:
answer

{'answer': 'Retrieval-Augmented Generation (RAG) reduces hallucinations in large language models by grounding responses in retrieved evidence from external sources. This allows models to use up-to-date and verifiable information beyond their fixed parameters, thereby mitigating the risk of generating false or misleading information. The context indicates that while RAG significantly reduces hallucinations, reliability remains a concern when the retrieved content is irrelevant, outdated, or misleading, as inaccuracies in retrieval can propagate into the final response.\n\nURL: https://arxiv.org/pdf/2610.01936',
 'status': 'success',
 'error': None,
 'token_usage': 2253,
 'chunks': [{'article_url': 'https://arxiv.org/pdf/2610.01936',
   'title': 'Mapping the RAG Landscape: A Four Axis Taxonomy of Efficiency, Defense, Interactivity, and Reasoning',
   'chunk_index': 108,
   'text': 'In deployment, this leads to outdated retrieval outputs, higher latency, and reduced reliability-problems t

## To do:
- ~~generate ground truth~~
- ~~test and record the results~~
- download 1 -2 more models and test the same flow with them. Chose the one that performed best
- With best model, fine-tune the system:
    - test different numbers of searched titles
    - test different number of downloaded articles
    - try different chunk sizes
    - try different number of top k in the chunk ranker
    - try a set of promot augmentation.
- modify the script to record the token usage
- add llm call to classify questions
- add general answers, no context

In [23]:
sys_evaluation_df = {
    "mean_latency": "",
    "mean_token_usage":"",
    "context_recall":"",
    "context_precision":"",
    "mean_faithfulness":""
}

In [24]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
golden_df = pd.read_csv(f"golden_dataset.csv")

In [25]:
results = []

for _, row in golden_df.iterrows():
    golden_id = row["golden_id"]
    question = row["question"]
    start = time.perf_counter()
    answer = rag.run(question)
    latency = time.perf_counter() - start
    results.append({
        "golden_id": golden_id,
        "answer": answer,
        "latency": latency
    })

evaluation_df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
evaluation_df.to_csv(f"test_run_output_{timestamp}.csv")


2026-10-06 16:18:07,664 | INFO | user prompt: How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?, web query: confidence supervision token reduction model efficiency
2026-10-06 16:18:20,344 | WARNING | arXiv timeout/no results for query: confidence supervision token reduction model efficiency
2026-10-06 16:18:21,314 | INFO | user prompt: How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?, web query: confidence supervision token reduction model efficiency
2026-10-06 16:18:21,314 | INFO | Retry query: confidence supervision token reduction model efficiency
2026-10-06 16:18:34,291 | WARNING | arXiv timeout/no results for query: confidence supervision token reduction model efficiency
Batches: 100%|██████████| 1/1 [00:00<00:00, 33.17it/s]
2026-10-06 16:18:34,421 | ERROR | RAG failed: Invalid URL 'timeout': No scheme supplied. Perhaps you meant https://timeout?
2026-10-06 16:18:35,444 | I

In [37]:
# pd.set_option("display.max_colwidth", None)
results = pd.DataFrame(results)
results.head(5)

golden_id  \
0          1   
1          2   
2          3   
3          4   
4          5   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

My first run with the golden question resulted in error, since the web queries that were generted by the llm were too specific, and the arxive website did returned no results:

```2026-10-05 18:43:55,370 | INFO | user prompt: How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?, web query: "confidence supervision token reduction 25% accuracy preservation Gemma Qwen Nemotron GPT-OSS"```

I have tried to resolve this error by prompt engineering and adding a retry query generation. If the first web qury has not found any titles, I prompt the llm to generate a new one and also mark the old query as "bad". In case if the second query has not generated anything, I stop retrying. After trying this approach I had 30% of failed test cases. After examining the failed test cases, I came to the conclusion that some of the generated questions from the golden data set are too specific and not suitable for arxive websearch. My solution to this problem is to let the llm answer the question without using the context. However, to restrict a user from using the RAG model for unrealted to its purpose prompts, I will add a classifier model, that would either let the prompt proceed further in the pipline or not. Only AI, CS related topics can be promt, or other topics that are represented in the arxive web-site.


In [27]:
failed = results[
    results["answer"].apply(lambda x: x["status"] == "failed")
]
print(f"Percentage of failed questions: {len(failed) / len(results) * 100}%")


Percentage of failed questions: 34.0%


In [28]:
results

golden_id  \
0           1   
1           2   
2           3   
3           4   
4           5   
5           6   
6           7   
7           8   
8           9   
9          10   
10         11   
11         12   
12         13   
13         14   
14         15   
15         16   
16         17   
17         18   
18         19   
19         20   
20         21   
21         22   
22         23   
23         24   
24         25   
25         26   
26         27   
27         28   
28         29   
29         30   
30         31   
31         32   
32         33   
33         34   
34         35   
35         36   
36         37   
37         38   
38         39   
39         40   
40         41   
41         42   
42         43   
43         44   
44         45   
45         46   
46         47   
47         48   
48         49   
49         50   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [29]:
logs = pd.read_csv("rag_logs.csv")

retry_then_warning = (
    logs["message"].str.contains("Retry", case=False, na=False)
    & logs["level"].shift(-1).eq("WARNING")
)
retry_then_success = (
    logs["message"].str.contains("Retry", case=False, na=False)
    & ~logs["level"].shift(-1).eq("WARNING")
)

print(f"In {retry_then_warning.sum()} test cases prompt retry did not work")
print(f"In {retry_then_success.sum()} test cases prompt retry worked")



In 15 test cases prompt retry did not work
In 1 test cases prompt retry worked


My solution with retrying worked only for one test case.

In [39]:
# pd.set_option("display.max_colwidth", None)
golden_df[golden_df["golden_id"].isin(failed["golden_id"])][["question"]].head(3)

,question
0,"How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?"
2,How does static compact doc alone affect issue resolution in real repos vs. source present?
3,"How does OC‑GS refine angles, keep axis, boostPSNR amid uneven rotation, dropped frames, sparse caps?"


In [31]:
# pd.set_option("display.max_colwidth", None)
answers = results[results["answer"].apply(lambda x: x["answer"] is not None)][["golden_id", "answer"]].copy()

answers["answer"] = answers["answer"].apply(lambda x: x["answer"])

combined = (golden_df[~golden_df["golden_id"]
            .isin(failed["golden_id"])][["golden_id", "question"]]
            .merge(answers,on="golden_id"))

combined[["question", "answer"]].head(3)

,question,answer
0,"How do authors minimize queries, achieve optimal tgt alignment, post‑process, prompting in black‑box generative AI?","Authors minimize queries and achieve optimal target alignment in black-box generative AI by developing post-processing algorithms that reduce the expected number of queries to the generator. These algorithms ensure that the joint attribute distribution of the generated outputs aligns as closely as possible with a user-specified target distribution. They demonstrate the optimality of these algorithms as the number of requested outputs $ m \to \infty $. Additionally, they introduce the Random Demand Coupon Collector (RDC) algorithm, which is a source-rate-free method that samples a target attribute sequence and then M-samples from the generator. The authors also highlight that post-processing methods, such as RDC and TA-RDC, can substantially improve alignment with the target attribute distribution even when input-side interventions like prompting fail. \n\nThe authors compare their post-processing methods with prompting-based interventions, showing that while prompting can be used to request specific attributes, it is not always effective due to models not following instructions. Post-processing methods, on the other hand, provide a more reliable way to align the generated outputs with the desired target distribution. \n\nThe experiments conducted on tasks such as text-to-image generation and geocoded persona generation demonstrate the effectiveness of these post-processing algorithms in improving statistical attribute alignment. \n\nURL: https://arxiv.org/pdf/2609.31607"
1,"How do teachers adapt conv. AI use, balancing repair, differentiation, translation, tech, learner, instr. tensions?","Teachers adapt conversational AI use by performing four modes of adaptive practices: repair, differentiation, translation, and balancing. These practices occur at the intersection of three tensions: technology, learners, and instruction. Teachers repair failing technology, differentiate for gaps in learners’ readiness, translate abstract AI concepts through familiar stories, and balance children’s creative agency with learning goals. The implementation of AI tools and curricula for elementary levels should be designed for teacher improvisation, including structured channels for reflection and design-iteration-reflection loops. This helps teachers navigate tensions and adapt the curriculum to fit their students’ needs.\n\nURL: https://arxiv.org/pdf/2609.31569"
2,How does DeepEdu‑v1’s long‑context engine yield TTFT speedup over vLLM in Vietnam AI tutoring?,"DeepEdu-v1’s long-context inference engine yields a TTFT (Time-to-First-Token) speedup over vLLM by amortizing token selection from per-sub-chunk to per-cluster granularity. This reduces retrieval calls by 7.7× on long-context retrieval, cutting prefill latency (TTFT) by roughly 35% while maintaining or improving task accuracy. In its deployed configuration, DeepEdu achieves a nearly 2× TTFT speedup over standard vLLM serving.\n\nURL: https://arxiv.org/pdf/2609.31568"


In [32]:
print(f"Mean end-to-end latency: {results["latency"].mean():.2f} sec")
print(f"{len(combined)} answers have been generated")

Mean end-to-end latency: 24.55 sec
33 answers have been generated


I will run the relevancy evaluation test for those 35 successfully generated answers.

In [33]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
combined.to_csv(f"questions-answers_{timestamp}.csv")

In [34]:
ques_ans_df = pd.read_csv("questions-answers.csv")
eval = Eval()
r = eval.answer_relevancy(ques_ans_df, 0.5)

/Users/vilya/Desktop/capstone_project/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-10-06 16:39:23,233 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


KeyboardInterrupt: 

2026-10-06 16:39:38,946 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
relevancy = pd.DataFrame(r)
relevancy.to_csv(f"relevancy_eval_results_{timestamp}.csv")


In [ ]:
pass_rate = (relevancy["score"] > 0.5).mean()
print(f"Relevancy pass rate: {pass_rate:.2%}")

In [ ]:
# f = eval.faithfullness(ques_ans_df)

In [ ]:
from datetime import datetime



## Evaluation
### System level:
1. Token usage per prompt.
2. [x] End-to-End Latency
3. Mean Answer Relevance — measures whether the generated answer is relevant to the expected answer. I will calculate the cosine similarity between the generated answer and the expected answer. Answers with a similarity score greater than 0.5 will be classified as relevant (`True`), while answers with a score of 0.5 or lower will be classified as irrelevant (`False`). I will then calculate the proportion of relevant answers among all generated answers.
4. Mean Context Precision — measures how much of the retrieved context is relevant to answering the question. It evaluates the proportion of relevant retrieved chunks among all retrieved chunks.
5. Faithfulness — measures whether the generated answer is supported by the retrieved context and does not contain unsupported claims (hallucinations). I will use an LLM-as-a-judge to evaluate faithfulness.
### Retrieval level
- Context precision/ recall @ k
- Mean reciprocal rank (MRR)
- Normalized Discounted Cumulative Gain (NDCG)  how well top-k is sorted


